In [5]:
from backend_langgraph.db import get_vectorstore, get_relevant_context, add_to_vector_database, has_relevant_data
from backend.llm import call_llm
from backend.util import is_generic
from backend.search import search_and_crawl

ModuleNotFoundError: No module named 'util'

In [7]:
from langgraph.graph import START, END, StateGraph
class AgentState:
    def __init__(self, prompt: str = None):
        self.prompt = prompt
        self.vectorstore = None
        self.llm = None
        self.context = None
        self.documents = None
        self.response = None
        self.needs_web = None
        self.context_exists = None

In [8]:
def initialize(state: AgentState) -> AgentState:
    print("Initializing agent state...")
    state.vectorstore = get_vectorstore()
    return state

def check_prompt(state: AgentState) -> AgentState:
    print(f"Prompt received: {state.prompt}")
    return state

def check_needs_web_search(state: AgentState) -> AgentState:
    state.needs_web = is_generic(state.prompt)
    print("Is the prompt generic:", state.needs_web)
    return state

def check_context_in_db(state: AgentState) -> AgentState:
    state.context_exists = has_relevant_data(state.vectorstore, state.prompt)
    print("Context exists in DB:", state.context_exists)
    return state

def retrieve_context(state: AgentState) -> AgentState:
    context_list = get_relevant_context(state.vectorstore, state.prompt)
    state.context = "\n\n".join(context_list) if context_list else None
    print("Retrieved context:", state.context)
    return state

async def perform_web_search(state: AgentState) -> AgentState:
    state.documents = await search_and_crawl(state.prompt)
    print("Web search documents:", state.documents)
    return state

def add_documents_to_db(state: AgentState) -> AgentState:
    add_to_vector_database(state.vectorstore, state.documents)
    print("Documents added to DB")
    return state

def call_llm_state(state: AgentState) -> AgentState:
    state.response = call_llm(state.llm, state.prompt, context=state.context)
    print("LLM Response:", state.response)
    return state

def finish(state: AgentState) -> AgentState:
    print("Final state reached.")
    return state


In [ ]:
workflow = StateGraph(AgentState)
workflow.add_transition(START, initialize)
workflow.add_transition(initialize, check_prompt)
workflow.add_transition(check_prompt, check_needs_web_search)
# Branch: if no web search needed, call LLM immediately.
workflow.add_transition(
    check_needs_web_search,
    call_llm_state,
    condition=lambda s: not s.needs_web
)
# Branch: if web search needed, check for context in DB.
workflow.add_transition(
    check_needs_web_search,
    check_context_in_db,
    condition=lambda s: s.needs_web
)
# If context exists, retrieve it.
workflow.add_transition(
    check_context_in_db,
    retrieve_context,
    condition=lambda s: s.context_exists
)
# If no context exists, perform web search.
workflow.add_transition(
    check_context_in_db,
    perform_web_search,
    condition=lambda s: not s.context_exists
)
# When web search completes, add documents to DB.
workflow.add_transition(perform_web_search, add_documents_to_db)
# After adding to DB, retrieve context.
workflow.add_transition(add_documents_to_db, retrieve_context)
# In both branches (retrieve_context or direct call) we call the LLM.
workflow.add_transition(retrieve_context, call_llm_state)
# Then finish.
workflow.add_transition(call_llm_state, finish)
workflow.add_transition(finish, END)


In [1]:
from langgraph.graph import StateGraph
from graphviz import Digraph
from IPython.display import display

def visualize_workflow(workflow: StateGraph):
    dot = Digraph(format="png")
    dot.attr(dpi="300")  # Increase resolution for clarity

    # Add nodes (states)
    for state in workflow.nodes:
        label = state.__name__ if callable(state) else state  # Use function name
        dot.node(label, shape="ellipse" if state != END else "doublecircle")

    # Add transitions (edges)
    for source, edges in workflow.edges.items():
        source_label = source.__name__ if callable(source) else source
        for target, condition in edges:
            target_label = target.__name__ if callable(target) else target
            edge_label = condition.__name__ if condition else ""  # Show condition name if available
            dot.edge(source_label, target_label, label=edge_label)

    return dot

# Generate the visualization
workflow_diagram = visualize_workflow(workflow)

# Display in Jupyter Notebook
display(Image(workflow_diagram.render("workflow_diagram", format="png")))


NameError: name 'workflow' is not defined

In [ ]:
# --- Run the Workflow ---

# Define an async runner since one state is async.
async def run_workflow(test_prompt: str):
    # Create initial state with the prompt.
    state = AgentState(prompt=test_prompt)
    # Run the state graph.
    final_state = await workflow.run_async(state)
    return final_state

In [ ]:
# Execute the workflow with a test prompt.
test_prompt = "Please search Google for the latest tech news"
final_state = asyncio.run(run_workflow(test_prompt))

print("\n--- Final Agent State ---")
print("Prompt:", final_state.prompt)
print("LLM Response:", final_state.response)